# 03 — Qwen3-8B campaign results analysis

Load one or more **immutable** run directories or `.zip` bundles, **verify every
hash before analysis**, reproduce the committed aggregate from the raw outcomes,
then emit the preregistered paired statistics, publication-quality tables, plots
(where a plotting library is present) and a machine-readable summary.

This notebook is an **interface**: every statistic is computed by
`benchmark/analysis.py` / `scripts/bench_analyze.py`. No scoring, pairing or
correction is written in a cell.

It runs unchanged on **CPU against the committed scripted run**
`benchmark/runs/20261008T203656Z-m6-campaign/` — the default parameters below.
That run is scripted replay (evidence class 1); this notebook makes no real-model
claim, and a real-model campaign run is a separate, not-yet-existing artifact.


In [ ]:
import json, os, sys, tempfile, zipfile, importlib.util
from pathlib import Path


def _repo_root(start):
    for candidate in (start, *start.parents):
        if (candidate / "scripts" / "bench_validate.py").is_file():
            return candidate
    return start


REPO = Path(os.environ.get("AEGISGRAPH_REPO") or _repo_root(Path.cwd())).resolve()
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from benchmark.analysis import (  # noqa: E402  (the tested analysis library)
    AnalysisError,
    analyse_campaign,
    load_run_directory,
    reproduce_score,
    verify_run_hashes,
    write_analysis_outputs,
)

print("repository:", REPO)
print("analysis library:", REPO / "benchmark" / "analysis.py")


In [ ]:
# @title Parameters
# Default: the committed scripted run, so this notebook executes on CPU without a model.
COMMITTED_RUN = REPO / "benchmark" / "runs" / "20261008T203656Z-m6-campaign"

# One or more run directories, campaign-bundle directories, or `.zip` bundles.
RUN_DIRS = [str(COMMITTED_RUN)]
# Optional allow-all control run directories (one shared, or one per run). A run
# directory that already carries its own `control.jsonl` needs no entry here.
CONTROL_DIRS = []
# The exact `.zip` bundles the campaign CLI produced (recorded for provenance).
BUNDLES = []

OUT_DIR = Path(os.environ.get("AEGISGRAPH_ANALYSIS_OUT", str(REPO / "analysis-out")))
GROUP_BY = ("domain", "attack_family")   # slice groupings: domain, attack_family
EXPLORATORY = False                      # True for the RQ3 ablation family (BH, q=0.05)
BOOTSTRAP_REPLICATES = 10000
FORCE = True                             # overwrite a non-empty OUT_DIR
EXPORT_PDF = False                       # also export PDF where a LaTeX toolchain exists
print("output directory:", OUT_DIR)


## 1. Resolve the inputs

A *bundle* is either a per-run `.zip` (the campaign CLI's `--bundle` output) or a
directory of run directories (the campaign bundle). Each is expanded to the run
directories it contains; no file is ever modified.


In [ ]:
_work = Path(tempfile.mkdtemp(prefix="aegisgraph-analysis-"))


def _is_run_dir(path):
    return (path / "manifest.json").is_file() and (path / "outcomes.jsonl").is_file()


def _expand(path):
    if path.is_dir() and _is_run_dir(path):
        return [path]
    if path.is_dir():
        return sorted(p for p in path.iterdir() if p.is_dir() and _is_run_dir(p))
    if path.is_file() and path.suffix == ".zip":
        target = _work / path.stem
        target.mkdir(parents=True, exist_ok=True)
        with zipfile.ZipFile(path) as archive:
            archive.extractall(target)
        if _is_run_dir(target):
            return [target]
        return sorted(p for p in target.iterdir() if p.is_dir() and _is_run_dir(p))
    raise FileNotFoundError(f"not a run directory, campaign bundle or .zip bundle: {path}")


RESOLVED_RUNS = [p for item in RUN_DIRS for p in _expand(Path(item))]
RESOLVED_CONTROLS = [p for item in CONTROL_DIRS for p in _expand(Path(item))]
if not RESOLVED_RUNS:
    raise FileNotFoundError("no run directories resolved from RUN_DIRS")
print("run directories:", [p.name for p in RESOLVED_RUNS])
print("control directories:", [p.name for p in RESOLVED_CONTROLS] or "(none)")


## 2. Verify every hash, then reproduce the committed score

`verify_run_hashes` re-hashes the recorded artifacts and `reproduce_score`
recomputes the aggregate from the raw outcomes and demands it equals the
committed `score.json`. A mismatch raises; it is never warned about and never
silently dropped.


In [ ]:
VERIFIED = {}
for run in RESOLVED_RUNS:
    verified, source = verify_run_hashes(run)
    loaded = load_run_directory(run, verify_hashes=True)
    reproduce_score(loaded)
    VERIFIED[run] = {"loaded": loaded, "hash_source": source, "files": sorted(verified)}
    print(f"verified {run.name}: {len(verified)} file(s) from {source}; score.json reproduced")


## 3. Paired analysis

`analyse_campaign` pairs each treatment run with a control (its own recorded
`control.jsonl` unless a control directory is given), applies the preregistered
corrections and writes `statistics.json`, the Markdown/CSV tables and any plots.


In [ ]:
try:
    result = analyse_campaign(
        [str(p) for p in RESOLVED_RUNS],
        control_dirs=[str(p) for p in RESOLVED_CONTROLS] or None,
        group_by=GROUP_BY,
        bootstrap_replicates=BOOTSTRAP_REPLICATES,
        exploratory=EXPLORATORY,
    )
    written = write_analysis_outputs(result, OUT_DIR, force=FORCE)
except AnalysisError as error:
    raise RuntimeError(f"analysis failed: {error}") from error

failed = [row for row in result["accounting"]["runs"] if row["status"] != "analysed"]
if failed:
    raise RuntimeError(f"{len(failed)} run(s) failed verification or reproduction: {failed}")
print(f"wrote {len(written)} artifact(s) into {OUT_DIR}")
print("statistics digest:", result["digest"])


## 4. Publication tables

In [ ]:
for table in sorted((OUT_DIR / "tables").glob("*.md")):
    print("=" * 78)
    print(table.name)
    print("=" * 78)
    print(table.read_text(encoding="utf-8"))


## 5. Plots and machine-readable summary

The forest plot is rendered by the analysis library when a plotting library is
importable; its absence is not an error. `statistics.json` **is** the
machine-readable summary.


In [ ]:
plots = result.get("plots") or []
if plots:
    for name in plots:
        print("plot:", OUT_DIR / "plots" / name)
else:
    print("no plots: a plotting library (matplotlib) is not importable in this runtime.")


In [ ]:
summary = {
    "statistics_json": str(OUT_DIR / "statistics.json"),
    "schema_version": result["schema_version"],
    "digest": result["digest"],
    "runs": [
        {"name": row["name"], "status": row["status"], "hash_source": row.get("hash_source")}
        for row in result["accounting"]["runs"]
    ],
    "comparisons": [
        {
            "treatment": comparison["treatment"],
            "reached_n": comparison["reached_n"],
            "control_asr": comparison["control_asr"],
            "treatment_asr": comparison["treatment_asr"],
            "reduction_absolute": comparison["reduction_absolute"],
            "exact_p": comparison["test"]["exact_p"],
            "adjusted_p": comparison["test"]["adjusted_p"],
            "family_m": comparison["test"]["m"],
        }
        for comparison in result["comparisons"]
    ],
}
print(json.dumps(summary, indent=2, sort_keys=True))


## 6. HTML / PDF export (where the runtime supports it)

Export is delegated to `nbconvert`; when it (or a LaTeX toolchain for PDF) is
absent the notebook still produces `statistics.json` and the Markdown/CSV tables.


In [ ]:
notebook_path = Path(os.environ.get("AEGISGRAPH_NOTEBOOK", str(REPO / "notebooks" / "03_qwen_results_analysis.ipynb")))

if importlib.util.find_spec("nbconvert") is None or not notebook_path.is_file():
    print("nbconvert is not available (or the notebook file is not present); skipping export.")
    print("The machine-readable deliverables are under", OUT_DIR)
else:
    import subprocess
    base = [sys.executable, "-m", "nbconvert", "--output-dir", str(OUT_DIR), str(notebook_path)]
    html = subprocess.run(base + ["--to", "html"], cwd=str(REPO), text=True)
    print("HTML export exit:", html.returncode, "->", OUT_DIR / (notebook_path.stem + ".html"))
    if EXPORT_PDF:
        pdf = subprocess.run(base + ["--to", "pdf"], cwd=str(REPO), text=True)
        print("PDF export exit:", pdf.returncode)
